# Thí nghiệm Big Data: Multimodal Fake News Detection (Fakeddit Subset)
Notebook này được thiết kế tự động để chạy trên Google Colab nhằm chứng minh năng lực của mô hình Cross-Attention trên tập dữ liệu lớn (Big Data).

## Bước 1: Khởi tạo môi trường và Clone mã nguồn
Tận dụng lại toàn bộ module M2-M5 đã được xây dựng chuẩn mực trên GitHub.

In [ ]:
!git clone https://github.com/btsuu25-dev/multimodal-fake-news-detection.git
%cd multimodal-fake-news-detection
!pip install -r requirements.txt
!pip install datasets tqdm requests pandas

## Bước 2: Tải và Tiền xử lý 30.000 mẫu Fakeddit (Subset)
Sử dụng thư viện `datasets` của HuggingFace để tải dữ liệu, sau đó dùng đa luồng (multi-threading) để tải nhanh 30.000 hình ảnh.

In [ ]:
import os
import requests
import pandas as pd
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. Tải Dataset Fakeddit từ HuggingFace (Chỉ lấy phần dữ liệu text và metadata)
print("Đang tải siêu dữ liệu Fakeddit...")
dataset = load_dataset("jako/Fakeddit")['train']
df = dataset.to_pandas()
df = df[df['hasImage'] == True] # Lọc các bài có ảnh
df = df[['image_url', 'clean_title', '2_way_label']].dropna().head(40000)

os.makedirs('data/images', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)

valid_rows = []
MAX_SAMPLES = 30000

# 2. Hàm tải ảnh
def download_image(row):
    if len(valid_rows) >= MAX_SAMPLES: return
    url = row['image_url']
    img_name = url.split('/')[-1]
    img_path = os.path.join('data/images', img_name)
    
    if not os.path.exists(img_path):
        try:
            res = requests.get(url, timeout=5)
            if res.status_code == 200:
                with open(img_path, 'wb') as f:
                    f.write(res.content)
                valid_rows.append({
                    'image_path': img_path,
                    'text': row['clean_title'],
                    'label': row['2_way_label']
                })
        except:
            pass

# 3. Tiến hành tải ảnh đa luồng siêu tốc
print(f"Đang tải {MAX_SAMPLES} ảnh... Quá trình này mất khoảng 5-10 phút.")
rows_list = [row for _, row in df.iterrows()]
with ThreadPoolExecutor(max_workers=32) as executor:
    list(tqdm(executor.map(download_image, rows_list), total=len(rows_list)))

print(f"\nĐã tải thành công {len(valid_rows)} mẫu dữ liệu hợp lệ!")

## Bước 3: Chia tập dữ liệu (Train/Val/Test)

In [ ]:
from sklearn.model_selection import train_test_split

final_df = pd.DataFrame(valid_rows)

# Chia theo tỷ lệ 80-10-10
train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=42, stratify=final_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_df.to_csv('data/splits/train.csv', index=False)
val_df.to_csv('data/splits/val.csv', index=False)
test_df.to_csv('data/splits/test.csv', index=False)

print("Đã lưu các file phân chia dữ liệu vào thư mục data/splits/")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Bước 4: Huấn luyện Mô hình Concat (Baseline)
Cùng xem sức mạnh của kiến trúc đơn giản khi gặp Big Data.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train_baseline.py

## Bước 5: Huấn luyện Mô hình Cross-Attention (Mô hình chính)
Kỳ vọng: Cross-Attention sẽ phát huy sức mạnh bắt chéo đặc trưng trên tập dữ liệu đa dạng và lớn hơn này.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'
!python src/train.py

## Bước 6: Đánh giá và Xuất báo cáo (HTML)

In [ ]:
!python src/evaluate.py
!python src/evaluation/html_report.py
print("Hoàn tất! Bạn có thể tải file results/dashboard.html về máy tính để xem kết quả so sánh cuối cùng!")